In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from model_functions import dataFullTransformer
import pickle

In [3]:
RANDOM_STATE = 47

In [4]:
df = pd.read_csv('data/zbiór_2.csv')

In [5]:
y = df['default']
df.drop(columns=['default'], inplace=True)

# Podział danych

In [6]:
X_train, X_test, y_train, y_test = train_test_split(df, y, random_state=RANDOM_STATE, stratify=y, test_size=0.4)

In [7]:
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, random_state=RANDOM_STATE, test_size=0.5, stratify=y_test)

In [8]:
X_val, X_cal, y_val, y_cal = train_test_split(X_val, y_val, random_state=RANDOM_STATE, test_size=0.5, stratify=y_val)

In [9]:
df_train = pd.concat([X_train, y_train], axis=1)
df_test = pd.concat([X_test, y_test], axis=1)
df_val = pd.concat([X_val, y_val], axis=1)
df_cal = pd.concat([X_cal, y_cal], axis=1)

df_train.to_csv('data/df_train_raw.csv', index=False)
df_test.to_csv('data/df_test_raw.csv', index=False)
df_val.to_csv('data/df_val_raw.csv', index=False)
df_cal.to_csv('data/df_cal_raw.csv', index=False)

60% train, 20% test, 10% kalibracja, 10% walidacja (mało danych, taka konieczność podziału)

In [10]:
print(X_train.shape, X_test.shape, X_val.shape)

(1800, 219) (600, 219) (300, 219)


In [11]:
print(y_train.shape, y_test.shape, y_val.shape)

(1800,) (600,) (300,)


# Uzupełnienie braków danych i inne przekształcenia

In [12]:
transformer = dataFullTransformer(X_train.columns)

In [13]:
numeric_pipe = transformer.transformers[1][1]

In [14]:
numeric_pipe

,steps,"[('replaceInf', ...), ('replace0missings', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,func,<function rep...001DA292BDB20>
,inverse_func,None
,validate,False
,accept_sparse,False
,check_inverse,True
,feature_names_out,'one-to-one'
,kw_args,None


Ważny jest podział danych przed transformacją danych, ponieważ fitujemy część Transformerów [zatem fit na całym zbiorze byłby wyciekiem danych]

In [15]:
np_prep_train = transformer.fit_transform(X_train, y_train)
X_train = pd.DataFrame(np_prep_train, columns=transformer.get_feature_names_out())

In [16]:
with open(r'models/dataPreprocessor.pkl', 'wb') as f:
    pickle.dump(transformer, f)

In [17]:
X_train.isna().mean().sort_values()

numeric_pipeline__-wsk_liczba_dni_istnienia          0.0
numeric_pipeline__-log_wsk_rent_kapitalu             0.0
numeric_pipeline__wsk_rent_kapitalu_sqrt             0.0
numeric_pipeline__wsk_stopa_zysku_sprzedaz_sqrt      0.0
numeric_pipeline__log_wsk_koszt_długu_1              0.0
                                                    ... 
numeric_pipeline__-wsk_zysk_ebitda_1                 0.0
numeric_pipeline__wsk_zast_kapitalu_podstawowego     0.0
numeric_pipeline__-wsk_struktura_kap_wlasnego_s_1    0.0
numeric_pipeline__wsk_rotacja_aktywow_1              0.0
symbols_pipeline__formaWlasnosci_Symbol              0.0
Length: 187, dtype: float64

In [18]:
X_train.shape

(1800, 187)

In [19]:
df_train = pd.concat([X_train, y_train], axis=1)

In [20]:
np_prep_test = transformer.transform(X_test)
X_test = pd.DataFrame(np_prep_test, columns=transformer.get_feature_names_out())
df_test = pd.concat([X_test, y_test], axis=1)

In [21]:
np_prep_val = transformer.transform(X_val)
X_val = pd.DataFrame(np_prep_val, columns=transformer.get_feature_names_out())
df_val = pd.concat([X_val, y_val], axis=1)

In [22]:
np_prep_cal = transformer.transform(X_cal)
X_cal = pd.DataFrame(np_prep_cal, columns=transformer.get_feature_names_out())
df_cal = pd.concat([X_cal, y_cal], axis=1)

Zapis danych do plików

In [23]:
df_train.to_csv('data/df_train_cleaned.csv', index=False)
df_test.to_csv('data/df_test_cleaned.csv', index=False)
df_val.to_csv('data/df_val_cleaned.csv', index=False)
df_cal.to_csv('data/df_cal_cleaned.csv', index=False)

In [24]:
df_train.corr()['default'].sort_values(ascending=False)

default                                                       1.000000
numeric_pipeline__log_wsk_zadluzenia                          0.220117
numeric_pipeline__-log_Kapital_zapasowy                       0.164034
numeric_pipeline__wsk_ogolnego_zadluzenia_pozyczki_squared    0.155354
numeric_pipeline__-log_wsk_ebitda_3                           0.150913
                                                                ...   
numeric_pipeline__wsk_Zobowiazania_krotkoterminowe_squared    0.001667
numeric_pipeline__wsk_zadluzenia_gotowki_1                    0.001551
numeric_pipeline__-log_wsk_plynnosc_gotowkowa_1               0.001237
numeric_pipeline__-wsk_obrotowosc_gotowkowa_squared           0.000820
numeric_pipeline__wsk_ebitda_koszty_finansowe_1_squared       0.000777
Name: default, Length: 188, dtype: float64